# Exploring ways to model "Who Pays" distribution more accurately

The Problem:
- France's insurance system decides based on a relative distruction basis which actor (farmer, government, insurance) covers how much of the damage.
- Until now we have fed the expected annual impact (EAI) into a function to estimate who has to pay how much. We realized however that this is not accuracte at all as the EAI has to be above the threshold of 20% so that the computation makes it seem as if the insurance is paying anything at all. The idea of insurance though is not to cover expected big damages but extreme events that do not happen like this every year.

What we would need to do, to make the estimate accurate:

- Compute every how many years the 20% and 50% boundary would be triggered (per hazard or maybe there is a way to make a single hazard object)
- Frequency * Who Pays * ? (It doesn’t make sense to multiply by EAI here because more of the damage will be in the upper categories)

How to actually do:
- Hazard * Impact Function -> Relative Impacts per Pixel (how does an impact object look like?
- Make a new Hazard object out of the impact object that just mixes the hazards (all relative impacts)
- Compute Yearly Impacts (here we would need to implement a multiplication of the relative impacts.)
- Then use the years to find out how often certain damages happen

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
## Imports

import sys
sys.path.append("..") # Adds the project root to the path

---

# Trying to Create Combined Hazard Object

In [3]:
from src.data_hazard import get_TC, get_TP

hazard_dict_TC = get_TC()
hazard_dict_TP = get_TP()

In [4]:
from src.data_exposure import get_exposure
from climada.entity import Exposures
import climada.util.lines_polys_handler as u_lp
from copy import deepcopy

exposure_poly = get_exposure(["TC", "TP", "MG"])

exposure_pnt = u_lp.exp_geom_to_pnt(
    exposure_poly,
    res=1000,
    to_meters=True,
    disagg_met=u_lp.DisaggMethod.FIX,
    disagg_val=None,
)

## Poly -> Eigen Raster Exposure
exposure_pnt_eigen_gdf = deepcopy(exposure_pnt.gdf)
exposure_pnt_eigen_gdf["value"] = 1
exposure_pnt_eigen = Exposures(exposure_pnt_eigen_gdf)

In [5]:
from climada.engine import ImpactCalc

impact_TC = ImpactCalc(
        exposure_pnt_eigen,
        impfset=hazard_dict_TC["impf_set"],
        hazard=hazard_dict_TC["hazard"]
    ).impact(save_mat=True)

impact_TC

In [6]:
from climada.engine import ImpactCalc

impact_TP = ImpactCalc(
        exposure_pnt_eigen,
        impfset=hazard_dict_TP["impf_set"],
        hazard=hazard_dict_TP["hazard"]
    ).impact(save_mat=True)

impact_TP

In [7]:
## Looking at what the Impact Matrix looks like
impact_TC.imp_mat[:10, :10].toarray()

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [8]:
## Testing how to extreact event's frequency
impact_TC.frequency

array([0.02857143, 0.02857143, 0.02857143, ..., 0.02857143, 0.02857143,
       0.02857143])

In [9]:
impact_TC.coord_exp

array([[45.62433544,  5.61991363],
       [45.62433544,  5.62889678],
       [45.62433544,  5.63787994],
       ...,
       [49.22373296,  1.71659393],
       [49.22373296,  1.72557709],
       [49.22373296,  1.73456024]])

## Build new Hazard

In [10]:
from scipy.sparse import vstack
import numpy as np

imp_mat_stacked = vstack([
    impact_TC.imp_mat,
    impact_TP.imp_mat
])

freq_stacked = np.concatenate([
    impact_TC.frequency,
    impact_TP.frequency,
])

In [11]:
from climada.hazard.centroids import Centroids

coords = impact_TC.coord_exp

centroids = Centroids(
    lat=coords[:, 0],
    lon=coords[:, 1]
)

In [12]:
fraction_mat = imp_mat_stacked.copy()
fraction_mat.data[:] = 1.0

In [18]:
from climada.hazard import Hazard

n_events, n_points = imp_mat_stacked.shape

hazard_combined = Hazard(haz_type="MG")

hazard_combined.centroids = centroids

hazard_combined.intensity = imp_mat_stacked
hazard_combined.frequency = freq_stacked
hazard_combined.fraction = fraction_mat

## Adding Infos
hazard_combined.event_id = np.arange(1, n_events + 1)
hazard_combined.event_name = np.array([f"event_{i}" for i in hazard_combined.event_id])
hazard_combined.date = np.arange(n_events).astype("datetime64[D]")


# --- Metadata ---
hazard_combined.units = "rel"
hazard_combined.tag = {
    "description": "Merged Events into Relative Impacts"
}

hazard_combined

## Hazard -> Impact

> (via Eigenexposure + Eigenimpact)

In [14]:
from climada.entity import ImpactFunc
from climada.entity.impact_funcs import ImpactFuncSet

impf_eigen = ImpactFunc(
    id=1,
    name = "Total Precipitation Impact Function (ChatGPT)",
    intensity_unit="mm",
    haz_type=hazard_combined.haz_type,
    intensity=np.array([0, 1]),
    mdd=np.array([0, 1]),
    paa = np.ones(2)
)

impf_eigen_set = ImpactFuncSet([impf_eigen])

In [19]:
impact_combined = ImpactCalc(
        exposures=exposure_pnt_eigen,
        impfset=impf_eigen_set,
        hazard=hazard_combined
    ).impact(save_mat=True)

impact_combined

---

# Sampling Years to Aggregate Impact

In [20]:
import numpy as np
import climada.util.yearsets as yearsets
from climada.engine import Impact

sampled_years = list(range(1900, 2000))

# direct computation
yimp, sampling_vect = yearsets.impact_yearset(
    impact_combined, sampled_years, correction_fac=False,
)

2026-05-13 19:32:45,692 - climada.util.yearsets - WARNING - The frequencies of the different events are not equal. This can lead to distorted sampling if the frequencies vary significantly. To avoid this, please set `with_replacement=True` to sample with replacement instead.


In [21]:
import scipy.sparse as sp

sparse_matrix = impact_combined.imp_mat
summed_matrix = sp.vstack(
    [
        (
            sp.csr_matrix(1.-(1.-sparse_matrix[indices, :].toarray()).prod(axis=0))
            if len(indices) > 0
            else sp.csr_matrix((1, sparse_matrix.shape[1]))
        )
        for indices in sampling_vect
    ]
)
yimp.imp_mat = summed_matrix

yimp.coord_exp = impact_combined.coord_exp
yimp.at_event, yimp.eai_exp, yimp.aai_agg = (
    ImpactCalc.risk_metrics(yimp.imp_mat, yimp.frequency)
)

In [22]:
from climada.util.plot import plot_from_gdf

threshold_values = [.20, .50]
gdf_rp, title, cols = yimp.local_return_period(threshold_values, method="extrapolate_constant")
plot_from_gdf(gdf_rp, title, cols)

KeyboardInterrupt: 

## Open Questions

Once I know how often the threshold is crossed, I need to know how much of the damage is occuring. Or is that the same? or the integral of how often times how much of the damage is in there?

